In [22]:
import pandas as pd
in_path="第一节代码导出的数据.csv"
all_factor=pd.read_csv(in_path)
del all_factor["Unnamed: 0"]
del all_factor["1M"]
del all_factor["3M"]
del all_factor["5M"]
del all_factor["10M"]
del all_factor["close"]
all_factor.head()

,time,code,日内收益率,volume,turnover_ratio,pe_ratio,pb_ratio,ps_ratio,pcf_ratio,roe,net_profit_margin,inc_net_profit_year_on_year,net_operate_cash_flow,circulating_market_cap
0,2025-01-02,000001.XSHE,-0.026032,-2.256970,-2.112075,-1.565787,-0.063302,-0.074557,-1.354803,1.504460,1.638136,0.977693,-0.944657,1.731194
1,2025-01-02,000002.XSHE,-0.019310,0.531564,0.365432,0.487044,0.949528,1.495057,0.217562,0.802671,0.236440,0.658079,1.556820,-0.693255
2,2025-01-02,000063.XSHE,-0.060843,-0.785244,-0.811384,-1.022391,-1.139433,-0.702402,-0.887764,0.714947,1.283861,0.264707,-0.049719,0.706712
3,2025-01-02,000100.XSHE,-0.016064,-0.010651,-0.192007,-0.297862,-0.886226,-0.231518,-0.700948,-0.469321,0.167125,-0.653161,-0.447469,-0.350280
4,2025-01-02,000157.XSHE,-0.029499,0.918861,0.984809,1.231699,1.012830,0.396327,1.011529,0.057020,-0.918804,0.330269,0.537585,-1.021096


In [23]:
def getICSeries(factors):
    fname = factors.columns[2]
    icall= factors.groupby("time").apply(lambda x: x.corr()[fname]).reset_index()
    icall = icall.dropna().drop([fname], axis=1).set_index("time")
    return icall


In [24]:
def correlation(dataset, threshold):
    """
    剔除数据集中高相关性的冗余特征，规避多重共线性
    参数:
        dataset: pandas.DataFrame - 输入的因子数据集（每行一个样本，每列一个因子）
        threshold: float - 相关性阈值（如0.8），超过此值的因子对将被视为高度相关
    返回:
        dataset: pandas.DataFrame - 剔除高相关冗余因子后的数据集
    """
    col_corr = set()  # Set of all the names of deleted columns
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if (corr_matrix.iloc[i, j] >= threshold) and (corr_matrix.columns[j] not in col_corr):
                colname = corr_matrix.columns[i] # getting the name of column
                col_corr.add(colname)
                # print(colname, corr_matrix.columns[j])
                if colname in dataset.columns:
                    del dataset[colname] # deleting the column from the dataset
                    # print("删除特征：{}".format(colname))
    return dataset

In [25]:
ic_factor = all_factor.copy()

In [26]:
time = all_factor["time"].unique()
count = 0
all_ret = []
all_t = []
all_select_factor = pd.DataFrame()

while True:
    if count>len(time)-31:
        break
    split_data = ic_factor[(ic_factor["time"]<time[29+count])&(ic_factor["time"]>=time[count])]
    icall = getICSeries(split_data)

    sort_ic = pd.DataFrame(abs(icall.mean()))
    sort_ic = sort_ic.sort_values(by=0, ascending=False)
    sort_ic = sort_ic[sort_ic[0]>0.03]

    X = icall.copy()
    X = X[sort_ic.index]
    X = correlation(X, 0.8)

    select_factor = pd.DataFrame()
    select_factor["factor"] = [X.columns.values]
    select_factor["time"] = time[29+count]
    # select_factor["ic"] = [icall.mean()[i] for i in X.columns]
    all_select_factor = all_select_factor.append(select_factor)

    # 每个季度筛选一次
    count += 11

In [27]:
def transfer_time(x):
    # 转换为日期格式，自动加1天
    date = pd.to_datetime(x)
    next_trading_day = date + timedelta(days=1)
    # 转回字符串格式，和原代码格式一致
    return next_trading_day.strftime("%Y-%m-%d")

In [28]:
from datetime import timedelta
all_select_factor["time"] = all_select_factor["time"].apply(lambda x: transfer_time(str(x)))
all_select_factor

,factor,time
0,"[pb_ratio, circulating_market_cap, turnover_ra...",2025-02-21
0,"[pb_ratio, circulating_market_cap, net_profit_...",2025-03-08
0,"[pcf_ratio, net_profit_margin, net_operate_cas...",2025-03-25
0,"[circulating_market_cap, pcf_ratio, net_profit...",2025-04-10
0,"[circulating_market_cap, net_operate_cash_flow...",2025-04-25
0,"[net_operate_cash_flow, volume, pcf_ratio, pb_...",2025-05-15
0,"[circulating_market_cap, pb_ratio, volume, pe_...",2025-05-30
0,"[circulating_market_cap, inc_net_profit_year_o...",2025-06-17
0,"[inc_net_profit_year_on_year, pe_ratio, circul...",2025-07-02
0,"[inc_net_profit_year_on_year, ps_ratio, turnov...",2025-07-17
